In [1]:
import duckdb
import pandas as pd
import plotly.express as px

db = duckdb.connect(
    "C:/Users/Decss/Desktop/retail-transaction-analyses/Data/retail.db",
    read_only=True
)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

# Basket Analysis

Business context:
 - Marketing vill förstå köpbeteende i kundkorgar.
    - Vad driver högt ordervärde?
    - Hur kan vi öka ordervärde?



Notering: “Orders från staging.top_anomalies med anomaly_type = 'anomaly_customer' har filtrerats bort. Dessa representerar stora manuella justeringar, plattformsavgifter eller ovanligt stora kunder som snedvrider analysen.”

In [2]:
# Genomsnittlig basket-value year over year (1 år)
db.sql("""
       SELECT
       ROUND(SUM(total_price) / COUNT(DISTINCT invoiceno), 2) as avg_basket_value
       FROM retail_enriched
       WHERE total_price > 0
       AND is_credit_invoice = 0
       AND customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       )
       AND invoicedate BETWEEN '2010-12-09' AND '2011-12-09'
       """)

┌──────────────────┐
│ avg_basket_value │
│      double      │
├──────────────────┤
│           466.83 │
└──────────────────┘

För att förstå kundernas köpbeteende kopplat till korgarna så måste vi titta närmare på hur mycket de handlar för, vad de handlar och hur fördelningen ser ut.
Är det någon vara som står för en stor del av korgarnas värde? Eller är det jämt fördelat mellan varorna?

Vi har filtrerat ut ett par anomalikunder (anomaly_type = 'anomaly_customer') som snedvred datan med väldigt stora orders och stora returer. Det är ingen perfekt lösning men problemet blir någorlunda isolerat.

En varukorg senaste året har i genomsnitt kostat 466.83. Men i och med att verksamheten har utvecklats under året så bör vi först räkna om detta värde till ett månatligt värde för att bättre förstå den löpande verksamheten. Vi tittar även på medianen för att få en bättre bild av hur de flesta korgar ser ut och inkluderar tillhörande graf på spridningen. 

In [3]:
# Basket value: Mean, Median, Number of orders
df_basket = db.sql("""
    WITH basket AS (
        SELECT
            invoicedate,
            invoiceno,
            ROUND(SUM(total_price), 2) AS basket_value
        FROM retail_enriched
        WHERE is_credit_invoice = 0
        AND customerid NOT IN (
    SELECT DISTINCT customerid 
    FROM staging.top_anomalies 
    WHERE anomaly_type = 'anomaly_customer'
)
        GROUP BY 1, 2
        HAVING SUM(total_price) > 0
    )

    SELECT
        DATE_TRUNC('month', invoicedate) AS month,
        ROUND(AVG(basket_value), 2) AS avg_basket,
        ROUND(MEDIAN(basket_value), 2) AS median_basket,
        COUNT(*) AS n_baskets
    FROM basket
    GROUP BY 1
    ORDER BY 1
""").df()

In [13]:
fig_basket_monthly = px.line(
    df_basket,
    x='month',
    y=['avg_basket', 'median_basket'],
    title=''
)
fig_basket_monthly = px.line(
    df_basket,
    x='month',
    y=['avg_basket', 'median_basket'],
    title=''
)
fig_basket_monthly.write_html(
    "../HTML/bask_basket_monthly.html",
    include_plotlyjs='cdn'
)
fig_basket_monthly.show()

Efter att ha hämtat datan för värdet på korgarna och inspekterat mean och median så ser vi att det skiljer sig en hel del mellan de två måtten. Mediankorgen ligger närmare 300 än de ca 450 som medelvärdet signalerar.

Anledningen är såklart att vi har en hel del ordrar långt över 450 som drar upp medelvärdet trots att de flesta ordrar faktiskt ligger närmare 300. 


In [5]:
# Distinkta baskets för distribution-visualisering
df_basket_dist = db.sql("""WITH basket as (
                   SELECT
                   invoiceno,
                   SUM(total_price) as basket_value
                   FROM retail_enriched
                   WHERE is_credit_invoice = 0
                   AND customerid NOT IN (
                   SELECT DISTINCT customerid 
                   FROM staging.top_anomalies 
                   WHERE anomaly_type = 'anomaly_customer'
                   )
                   GROUP BY 1
                   HAVING SUM(total_price) > 0
                   )
                   SELECT
                   *
                   FROM basket
                   """).df()

In [6]:
# Histogram för korg-ditribution efter pris och antal
basket_fig = px.histogram(df_basket_dist, 
             x='basket_value',
             nbins=20,
             title='Basket Value Distribution')
basket_fig.update_traces(xbins=dict(start=0, end=5000, size=100))
basket_fig.write_html(
    "../HTML/bask_basket_v_distribution.html",
    include_plotlyjs='cdn'
)
basket_fig.show()

Här blir det tydligt hur fördelningen av korgvärde ser ut. De flesta av våra orders ligger på ett värde runt 0-400. Men är det dessa som driver intäkter? Vi kikar lite närmare på fördelningen av intäkter sett till ordervärden för att sedan ta en titt på hur dessa ordrar ser ut och vad det är som driver intäkterna.

In [7]:
# Revenue uppdelad i buckets

df_basket_uitem_dist = db.sql("""
                              WITH basket_uitem AS(
                              SELECT 
                              invoiceno,
                              SUM(total_price) as basket_value
                              FROM retail_enriched
                              WHERE is_credit_invoice = 0
                              AND customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              )
                              GROUP BY 1
                              HAVING SUM(total_price) > 0
                              )
                              
                              SELECT
                              CASE 
                                WHEN basket_value <= 100 THEN '0-100'
                                WHEN basket_value <= 200 THEN '101-200'
                                WHEN basket_value <= 300 THEN '201-300'
                                WHEN basket_value <= 400 THEN '301-400'
                                WHEN basket_value <= 500 THEN '401-500'
                                WHEN basket_value <= 750 THEN '501-750'
                                WHEN basket_value <= 1000 THEN '751-1000'
                                WHEN basket_value <= 1500 THEN '1001-1500'
                                WHEN basket_value <= 2000 THEN '1501-2000'
                                WHEN basket_value <= 5000 THEN '2001-5000'
                              ELSE '5000+'
                              END as basket_value_bucket,
                              CASE
                                WHEN basket_value <= 100 THEN 0
                                WHEN basket_value <= 200 THEN 101
                                WHEN basket_value <= 300 THEN 201
                                WHEN basket_value <= 400 THEN 301
                                WHEN basket_value <= 500 THEN 401
                                WHEN basket_value <= 750 THEN 501
                                WHEN basket_value <= 1000 THEN 751
                                WHEN basket_value <= 1500 THEN 1001
                                WHEN basket_value <= 2000 THEN 1501
                                WHEN basket_value <= 5000 THEN 2001 
                              ELSE 5001
                              END as bucket_sort,
                              ROUND(SUM(basket_value), 1) as total_revenue,
                              ROUND(SUM(basket_value)*1.0 / SUM(SUM(basket_value)) OVER(), 3) as revenue_share,
                              /* viktigt med 1.0 för att sätta "float" på värdet. Annars delas bara heltal */
                              ROUND(AVG(basket_value), 1) as avg_revenue,
                              COUNT(*) as n_orders
                              FROM basket_uitem
                              WHERE basket_value > 0
                              GROUP BY 1,2
                              ORDER BY 2
                              """).df()

Enstaka customer-anomalies har filtrerats bort för att ge en bättre bild av hur värde per order i stort sett är fördelat (enligt notering i intro). Filtreringen påverkar 5000+ segmentet negativt, och det går från att vara det största segmentet sett till intäkter till att var mer in line med andra segmenten (12,8%). (En stor del av det som har filtrerats bort inkluderar manuella justeringar, plattformskostnader och frakt) För basket-analysen är detta godtagbart då vi vill få en tydligare bild över helheten.

Exkluderat anomalierna ser vi att en stor del av vår försäljning sker vid ordervärden kring 301-400 (14,2%) och 501-750 (14%). 52% av försäljningen sker i ordrar värda 101-750.

Sett till antalet ordrar så representerar baskets över 2000 i värde en ganska liten andel, men i totala intäkter så är dessa relativt få ordrar väldigt betydande. 

In [8]:
# Kumulativ andel av försäljning (basket value buckets) 80/20 intäkter
df_basket_uitem_dist = df_basket_uitem_dist.sort_values('bucket_sort').reset_index(drop=True)
df_basket_uitem_dist['cumulative_share'] = df_basket_uitem_dist['revenue_share'].cumsum()

fig = px.bar(
    df_basket_uitem_dist,
    x='basket_value_bucket',
    y='cumulative_share',
    title='Cumulative Revenue Share per Basket Value Bucket',
    text='cumulative_share'
)
fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_yaxes(tickformat='.0%')
fig.write_html(
    "../HTML/bask_cumulative_share.html",
    include_plotlyjs='cdn'
)
fig.show()

75% av intäkterna kommer från orders 0-2000. redan vid 750 så har mer än 50% av intäkterna genererats. De viktigaste segmenten är däremot de som är närmare mitten, 301-1500 som står för ca 40%. 


In [9]:
# Unika varor i korgar (efter returer) per månad över alla ordervärden.
df_unique_items = db.sql("""
                         WITH items as (
                         SELECT
                         invoiceno,
                         MIN(invoicedate) as invoicedate,
                         COUNT(DISTINCT stockcode) as unique_items,
                         SUM(total_price) as basket_value
                         FROM retail_enriched
                         WHERE is_credit_invoice = 0
                         AND customerid NOT IN (
                         SELECT DISTINCT customerid 
                         FROM staging.top_anomalies 
                         WHERE anomaly_type = 'anomaly_customer'
                         )
                         GROUP BY 1
                         HAVING SUM(total_price) > 0
                         )

                         SELECT
                         DATE_TRUNC('month', invoicedate) as month,
                         ROUND(AVG(unique_items), 2) as avg_unique_items,
                         MEDIAN(unique_items) as median_unique_items
                         FROM items
                         WHERE basket_value > 0
                         GROUP BY 1
                         ORDER BY 1
                         
                         """).df()

Antal unika items per basket baskets fluktuerar marginellt under året och tyder på stabil variation i baskets under året. Jämför vi genomsnitt med median så hintar det däremot om att unika items i korgarna beror på orderns värde, vilket är rimligt. 
Stora orders inkluderar förmodligen fler unika items än små orders.
    
    > Vi tittar närmare på detta nedan.

In [10]:
# Unika items per basket-value-grupp
df_basket_items_by_bucket  = db.sql("""
                              WITH basket_uitem AS(
                              SELECT 
                              invoiceno,
                              COUNT(DISTINCT stockcode) as unique_items,
                              SUM(total_price) as basket_value
                              FROM retail_enriched
                              WHERE is_credit_invoice = 0
                              AND customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              )
                              GROUP BY 1
                              HAVING SUM(total_price) > 0
                              )
                              
                              SELECT
                              CASE 
                                WHEN basket_value <= 100 THEN '0-100'
                                WHEN basket_value <= 200 THEN '101-200'
                                WHEN basket_value <= 300 THEN '201-300'
                                WHEN basket_value <= 400 THEN '301-400'
                                WHEN basket_value <= 500 THEN '401-500'
                                WHEN basket_value <= 750 THEN '501-750'
                                WHEN basket_value <= 1000 THEN '751-1000'
                                WHEN basket_value <= 1500 THEN '1001-1500'
                                WHEN basket_value <= 2000 THEN '1501-2000'
                                WHEN basket_value <= 5000 THEN '2001-5000'
                              ELSE '5000+'
                              END as basket_value_bucket,
                              CASE
                                WHEN basket_value <= 100 THEN 0
                                WHEN basket_value <= 200 THEN 101
                                WHEN basket_value <= 300 THEN 201
                                WHEN basket_value <= 400 THEN 301
                                WHEN basket_value <= 500 THEN 401
                                WHEN basket_value <= 750 THEN 501
                                WHEN basket_value <= 1000 THEN 751
                                WHEN basket_value <= 1500 THEN 1001
                                WHEN basket_value <= 2000 THEN 1501
                                WHEN basket_value <= 5000 THEN 2001 
                              ELSE 5001
                              END as bucket_sort,
                              ROUND(AVG(unique_items)) as avg_unique_items,
                              MEDIAN(unique_items) as median_unique_items
                              FROM basket_uitem
                              GROUP BY 1,2
                              ORDER BY 2
                              """).df()

In [11]:
# Unique items tipping point
tipping_pt_fig = px.line(
    df_basket_items_by_bucket,
    x='basket_value_bucket',
    y='median_unique_items',
    title='Median unique items per value bucket'
)
tipping_pt_fig.write_html(
    "../HTML/bask_med_uitem.html",
    include_plotlyjs='cdn'
)
tipping_pt_fig.show()

Med unika items uppdelat på ordervärden så ser vi tydligt att värdet har en betydande roll i antalet unika items. avg_unique_items påvisar ett linjärt förhållande, förmodligen pådrivet av outliers inom grupperna. För som vi ser i median_unique_items så hintar det om att många ordrar inte består av mer än 35 unika items. 

De ordervärdes-segment som främst tycks uppleva störst variation (median) av items är 751-1000 (32 st) och 1001-1500 (35 st), men även vårat största segment, 501-750 (27 st), vilket är anmärkningsvärt. Det är upp till denna punkt som kunder överlag handlar brett från vårat sortiment, medans de i segmenten efter (1500+) börjar handla mer "bulk" av enskilda items (även om genomsnittet tyder på att några aktörer handlar brett i de övre värde-segmenten).

Vi tittar därav närmare på 501-750 segmentet, 2001-5000 segmentet för att se hur dessa baskets är uppbygda och hur de skiljer sig.
Vi tittar även på 301-400-segmentet som också är en betydande del av våra intäkter.

In [12]:
# Titta närmare på orders i bucket 301-400, 500-750 och 2001-5000
focus_baskets = db.sql("""
       WITH basket AS(
        SELECT
        invoiceno,
        SUM(total_price) as basket_value
        FROM retail_enriched
        WHERE is_credit_invoice = 0
        AND customerid NOT IN (
            SELECT DISTINCT customerid 
            FROM staging.top_anomalies 
            WHERE anomaly_type = 'anomaly_customer'
            )
       GROUP BY invoiceno
       ),

      labeled AS (
       SELECT
       r.*,
       b.basket_value,
       CASE
        WHEN b.basket_value BETWEEN 301 AND 400 THEN '301-400'
        WHEN b.basket_value BETWEEN 501 AND 750 THEN '501-750'
        WHEN b.basket_value BETWEEN 2001 AND 5000 THEN '2001-5000'
       END as segment
       FROM retail_enriched r
       JOIN basket b USING (invoiceno)
                       
       )

    SELECT
    segment,
    stockcode,
    description,
    SUM(total_price) as total_revenue,
    SUM(quantity) as total_quantity,
    COUNT(*) as n_lines
    FROM labeled
    WHERE segment IS NOT NULL
    AND stockcode NOT IN ('POST', 'M')
    AND description NOT ILIKE '%POSTAGE%'
    AND segment = '2001-5000'
    GROUP BY segment, stockcode, description
    ORDER BY total_revenue DESC 
    LIMIT 25
                       
    """).df()

Vid inspektion av de olika value-buckets så kan vi se att produkter som kunder köper inte skiljer sig särskilt mellan de olika order-värdena. Små, medelstora och stora orders innehåller i princip samma typ av varor. Men, mellan 0-1500 i ordervärde så kan vi se att produktmixen breddas med stigande ordervärde för att sedan avta efter 1500. Detta indikerar att ordervärde fram till 1500+ segmentet drivs av både volym och bredare inköp. Efter ~1500 avtar produktbredden samtidigt som ordervärdet fortsätter öka tillsammans med volymen. 

Mediankorgen har ett värde på 300, men den största delen av våra intäkter kommer från orders över 400 (se kumulativ revenue share). Vi har alltså ett stort antal orders runt och under 300 som vi skulle vilja pressa upp mot bucket 400 och över.

Verksamheten är till synes väldigt B2B-driven, speciellt i högre basket-value-segment. Top-produkterna här handlas kontinuerligt i volym vilket inte är ett beteende som speglar typiska retailkunder. Vi kommer därav att behöva erbjuda eller pusha på värde som tilltalar majoriteten av våra kunder som är B2B.

# Rekommendationer
* Cross-Selling
    - Segment 0-1500
Eftersom vi ser att segment upp till 1500 till viss del drivs av ökad produktbredd så skulle vi kunna erbjuda/föreslå cross-selling. Förslag på produkter som kunder med liknande korgar men i högre segment också köper. Detta till fördelaktiga priser som add-on på korgen innan utcheckning.

* BUNDLING
    - Segment 0-1500
Erbjuda bundles tillsammans med de produkter som pushas via cross-sell. Datan visar att kunder ordervärde inom dessa segment ökar tillsammans med produktbredd. 

    - Segment 1500+
Dessa kunder spenderar inte mer genom att öka bredden av sina korgar, utan spending ökar främst när volym av enskilda produkter ökar. Vi bör därav erbjuda dessa kunder möjligheten att köpa volymbundles. Vi vet att volym till bättre pris är attraktivt men att vi möjligtvis skulle kunna pusha bredare köp om vi kombinerar bundling med bättre volymjusterat pris. Bundling i detta fall bör nödvändigtvis inte vara discovery-orienterat, utan fokusera på produkter som vi vet att dessa kunder ofta köper.

* VOLUME INCENTIVE
    - Segment 1500+
Vi har identifierat att orders över detta segment främst drivs av volym. Vi bör därav se till att vi kan pusha för större orders genom fördelar som kunder kan ta del av om de beställer mer. Lägre unitpriser, rabatter eller erbjudanden när beställningar når specifika nivåer. 
Det skulle kunna vara intressant att unersöka om vi kan erbjuda procentuella rabatter över 1500 i olika tiers.
Öppna för möjlighet av individuell förhandling över viss volym.

* FREE SHIPPING
    - Segments 350+
Givet att medianvärdet på en varukorg är runt 300 och att distributionen även tyder på att orders klustrar kring medianvärdet så bör fri frakt strax över denna nivå övervägas.
Att erbjuda fri frakt runt 350-400 skulle kunna hjälpa tillatt pusha kunder från  301-400 segmentet in i 401-500 segmentet.
I kombination med bundles och cross selling så har vi goda möjligheter att pusha median spend uppåt.